# 一筆解析パイプライン — 圃場単位の特徴量データセットを作る（W7）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-sat-gis/blob/main/public/notebooks/tutorials/04_field_pipeline.ipynb)

> **Private リポジトリの注意:** 上のバッジから直接開けない場合は
> [Google Colab](https://colab.research.google.com/) で「ファイル → ノートブックを開く → GitHub → プライベート リポジトリを含める」から開いてください。

## 本ノートで何をするか

最終課題「相馬市・南相馬市の作付／休耕を判別する」の**実装上の最大の山場**、
「衛星データの**画像**」を「**一筆（圃場）ごとの特徴量の表**」に変換するパイプラインを組みます。

```text
 Sentinel-2 NDVI / Sentinel-1 VV・VH（dB）
               ↓
        農地ポリゴン（一筆）   ← 農水省「筆ポリゴン」
               ↓
   各筆について 月別の平均値（Zonal Statistics）
               ↓
   時系列の「波形特徴量」を計算（Python）
               ↓
  field_features.csv   ←「1筆×1年=1行」の解析データセット（W8で判別・精度評価に使う）
```

## 重要: 画像と表は別物

GEE で「きれいな NDVI 画像が出た！」で止まってしまうと、最終課題にはつながりません。
**最終的に必要なのは「表」**です。本ノートのゴールは「**1筆×1年=1行**」（`field_id` × `year` で1行）の表を作ることです。

| field_id | year | ndvi_max | ndvi_amplitude | vh_min | vv_vh_may | ... |
|----------|-----:|---------:|---------------:|-------:|----------:|----:|
| field_001 | 2023 |    0.78 |           0.50 |  -19.2 |     +8.2 | ... |
| field_002 | 2023 |    0.24 |           0.03 |  -13.1 |     +5.5 | ... |

（`vh_min` と `vv_vh_may` の単位は dB。`vv_vh_may` は VV と VH の**差**であり、
線形スケールでの VV/VH 比にあたる。）


## GEE と Python の役割分担

「全部 GEE でやるの？全部 Python でやるの？」で混乱しないように、役割を分けます。

| 担当 | 仕事 |
|------|------|
| **GEE（クラウド）** | Sentinel-1/2 の取得・時間フィルタ・雲除去・軌道方向統一・バンド選択・NDVI計算・SAR前処理・**空間集計（Zonal Statistics）**・Export |
| **Python（Colab）** | DataFrame化・**波形特徴量の整理**・可視化・統計・閾値分析・機械学習・混同行列・F1/P/R/IoU・結果CSV/GIS生成 |

> **画像の重い処理は GEE に、表の加工と解析は Python に。** このノートもこの分担に沿って進みます。


---

## 学習目標
- **Zonal Statistics**（`reduceRegions`）で、農地ポリゴンごとに月別の NDVI / VV / VH 平均を集計できる
- 「単一の画像」ではなく「**時系列の波形**」から特徴量を作る考え方を身につける
  - 休耕地は「NDVIが低い」ではなく「**年間を通じて生育ピークが形成されない**」
- 最終課題で使える **`field_features.csv`（1筆×1年=1行）** を完成させる


In [ ]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
!pip install -q geemap earthengine-api pandas scikit-learn matplotlib japanize-matplotlib geopandas


In [ ]:
# ---- ライブラリ読み込み ----
import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

try:
    import japanize_matplotlib  # あれば使う（無くても env.setup_font() が日本語対応）
except ImportError:
    japanize_matplotlib = None  # 未インストールでも続行（ローカルはシステムフォントで表示）
print('OK')

# ---- 共通環境ヘルパー（Colab / ローカル両対応） ----
try:
    from env import gee_setup, setup_font, data_dir
    gee_setup()          # GEE 認証・初期化（既存認証があればブラウザは開かない）
    setup_font()         # 日本語フォントを OS 別に設定
    DATA_DIR = data_dir()   # データ置き場（Colab=Drive / ローカル=tutorials/data）
except ImportError:
    print('（env.py が無いため、従来の Colab 用設定で続行します）')


## 再現性ヘッダー — 設定を1か所にまとめる

解析条件をコードの中に散らばせず、**この1ブロックに集約**します。
3週間後に自分（や他のメンバー）が同じ結果を再現できるようにするための仕組みです。

- 値を変えたときは、**このブロックの値も必ず書き換える**のがルールです
- 技術レポート §8「再現性」にはこのブロックをそのまま転記します
- 全教材で共通の設定です（`resources/product-related/guide_camp.md` §5 と同じ内容）


In [ ]:
# ============================================================
# 一筆解析パイプライン（W7/W8/合宿 共通）
# Notebook : 04_field_pipeline.ipynb
# Author   : Team A（氏名を全員分）
# Date     : 2026-08-29
# Version  : 1.0
#
# Input    : tutorials/data/fukushima_polygons.geojson（最大200筆）
# Output   : submissions/camp/team-*/field_features.csv
# ============================================================

# ---- データ取得の条件 ----
START_DATE  = '2021-01-01'            # 取得期間の開始
END_DATE    = '2025-12-31'            # 取得期間の終了
YEARS       = [2021, 2022, 2023, 2024, 2025]   # 1筆 × 1年 = 1行
MONTHS      = [3, 4, 5, 6, 7, 8, 9, 10]        # 季節ウィンドウ（月別平均）
TARGET_YEAR = 2024                    # 解析対象年（W8 の判別・精度評価で使う年）

# ---- 空間の条件 ----
CRS   = 'EPSG:32654'   # WGS84 / UTM zone 54N ← 変えるとポリゴンと画像がずれる
SCALE = 10             # m  reduceRegions の集計スケール（S1/S2 の native 解像度）

# ---- センサの条件 ----
S2_CLOUD_TH = 20            # %  CLOUDY_PIXEL_PERCENTAGE（シーン単位の雲量フィルタ）
S1_ORBIT    = 'DESCENDING'  # 軌道方向 ← ASC/DESC を混在させると時系列ノイズになる
#                            この地域は 2022-2024 に ASC の観測が無いため DESC を使う
S1_MODE     = 'IW'          # 観測モード（偏波 VV / VH）

# ---- 疑似ラベルの閾値（W8 でチームが決めた値をここに記録する） ----
VH_FLOOD_THRESH  = -18.0    # dB  5月の湛水判定（vh_may）
NDVI_HIGH_THRESH = 0.70     #     8月の作付判定（ndvi_aug）
NDVI_LOW_THRESH  = 0.25     #     年間最大が低い＝休耕（ndvi_max）
CLF_THRESHOLD    = 0.50     #     判別モデルの採用閾値

print(f'{START_DATE}〜{END_DATE} / {CRS} / scale={SCALE}m / '
      f'cloud<{S2_CLOUD_TH}% / S1={S1_ORBIT}')


In [ ]:
# 解析対象の範囲（02と同じ: 相馬市 + 南相馬市東側）
roi_soma   = ee.Geometry.Rectangle([140.90, 37.77, 141.01, 37.83])   # 相馬市
roi_minami = ee.Geometry.Rectangle([140.99, 37.60, 141.04, 37.66])   # 南相馬市 東側クラスタ
roi = roi_soma.union(roi_minami)
print('ROI を設定しました')


## 圃場ポリゴン（農地区画）を準備する

ここで使う「一筆」とは、農地ポリゴンの**ひとつの区画**のことです。
本番では **農林水産省の「筆ポリゴン」**（公開データ）を読み込みます
（入手手順は `resources/data-guide.md`）。**未ダウンロードでも動くよう、サンプルにフォールバック**します。

- 実データ（Colab）: `/content/drive/MyDrive/astrocamp-2026/export/farmland/fukushima_polygons.geojson`
- 実データ（ローカル）: `tutorials/data/fukushima_polygons.geojson`（動作確認用サンプル）
- パスは環境変数 `ASTROCAMP_POLYGON_PATH` で上書きできます
- フォールバック: 02 と同じサンプル矩形（相馬・南相馬の農地付近）


In [ ]:
# ---- 圃場ポリゴンを読み込む（実ポリゴン → なければサンプル矩形） ----
import os
try:
    from env import polygon_path
    POLY_PATH = polygon_path()   # Colab=Drive / ローカル=tutorials/data
except ImportError:
    POLY_PATH = '/content/drive/MyDrive/astrocamp-2026/export/farmland/fukushima_polygons.geojson'

SAMPLE_FIELDS = [
    ee.Feature(ee.Geometry.Rectangle([140.948, 37.787, 140.958, 37.797]), {'field_id': 'field_001'}),
    ee.Feature(ee.Geometry.Rectangle([140.960, 37.795, 140.968, 37.803]), {'field_id': 'field_002'}),
    ee.Feature(ee.Geometry.Rectangle([141.000, 37.636, 141.010, 37.644]), {'field_id': 'field_003'}),
    ee.Feature(ee.Geometry.Rectangle([140.996, 37.606, 141.006, 37.614]), {'field_id': 'field_004'}),
    ee.Feature(ee.Geometry.Rectangle([140.953, 37.782, 140.963, 37.792]), {'field_id': 'field_005'}),
    ee.Feature(ee.Geometry.Rectangle([140.966, 37.800, 140.974, 37.808]), {'field_id': 'field_006'}),
]

POLY_SOURCE = 'サンプル矩形（フォールバック）'
try:
    if os.path.exists(POLY_PATH):
        import geopandas as gpd
        gdf = gpd.read_file(POLY_PATH)
        gdf = gdf.cx[140.90:141.04, 37.60:37.83]        # ROI に重なる範囲に絞る
        gdf = gdf.head(200).reset_index(drop=True)      # 処理速度のため最大200筆
        gdf['field_id'] = [f'field_{i:03d}' for i in range(len(gdf))]
        fields = geemap.gdf_to_ee(gdf)                   # geopandas → GEE
        POLY_SOURCE = '農水省筆ポリゴン（実データ）'
        print(f'[ポリゴン] 実筆ポリゴン {len(gdf)} 筆を読み込みました')
    else:
        raise FileNotFoundError('ポリゴン未アップロード')
except Exception as e:
    print(f'[フォールバック] サンプル矩形を使います: {e}')
    fields = ee.FeatureCollection(SAMPLE_FIELDS)
    print(f'[ポリゴン] サンプル矩形 {len(SAMPLE_FIELDS)} 筆')

print('圃場ポリゴン数:', fields.size().getInfo(), '/ ソース:', POLY_SOURCE)
print('圃場 ID の例:', fields.limit(3).aggregate_array('field_id').getInfo())


## GEE でデータを準備する（S2 NDVI・S1 VV/VH）

03 と同じルールを守ります。
- Sentinel-2: `CLOUDY_PIXEL_PERCENTAGE < 20`（シーン単位）＋ SCL による画素単位の雲マスクの2段で雲を除去し、`normalizedDifference(['B8','B4'])` で NDVI
- Sentinel-1: IW モード・`orbitProperties_pass = DESCENDING` で**軌道方向を統一**（この地域は DESC しか通年で揃わない・下記）
- S1_GRD の VV/VH は**既に dB 値**なので、`10*log10` の二重変換をしない


In [ ]:
# ── Sentinel-2 NDVI と Sentinel-1 VV/VH（dB）のコレクションを準備 ──
def add_ndvi(img):
    return img.addBands(img.normalizedDifference(['B8', 'B4']).rename('NDVI'))

# 雲の除去は2段構え:
#   ① シーン単位 … CLOUDY_PIXEL_PERCENTAGE でシーンごと落とす（粗いふるい）
#   ② 画素単位   … SCL バンドで雲・影・cirrus の画素だけ抜く（細かいふるい）
# ①だけだと「雲量19%のシーン」の雲そのものが NDVI に混ざる。
try:
    from satellite_utils import s2_cloud_mask
except ImportError:
    print('[注意] satellite_utils.py が無いため画素単位の雲マスクを省略します')
    s2_cloud_mask = lambda img: img

s2_ndvi_all = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', S2_CLOUD_TH))   # ① シーン単位
    .map(s2_cloud_mask)                                             # ② 画素単位（SCL）
    .map(add_ndvi))

s1_all = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(roi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', S1_MODE))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('orbitProperties_pass', S1_ORBIT)))
# S1_GRD の VV/VH は既に dB。二重変換しない
s1_db_all = s1_all.map(lambda img: img.select(['VV', 'VH']).rename(['VV_db', 'VH_db']))

print('S2 scenes:', s2_ndvi_all.size().getInfo(), '/ S1 scenes:', s1_db_all.size().getInfo())
years  = YEARS
months = MONTHS   # 作付・生育・収穫までをカバーする月


## Zonal Statistics — 月別の平均値を「筆ごと」に集計する

ここが今回の山場です。**「一筆の中の画素を全部平均して、筆に1つの値を持たせる」** のが
Zonal Statistics（空間集計）です。GEE では `reduceRegions` で一気に行います。

```python
img.reduceRegions(collection=fields, reducer=ee.Reducer.mean(),
                  scale=10, crs='EPSG:32654')
#      ↑ 対象画像       ↑ 一筆ポリゴン   ↑ 筆内の平均   ↑ 画素サイズ  ↑ 集計する座標系
#
# scale と crs は必ず明示する。省略すると画像側の既定値で集計され、
# 「どの解像度・どの座標系で集計したか」がコードから読み取れなくなる。
```

月ごとに「その月の NDVI / VH / VV の平均画像」を作り、一括で筆に平均値を落とします。


## ✅ Check 1

今、1つのセルは何を表していますか？

> **1つの圃場の1か月の観測値**（月別平均 NDVI / VH / VV）

`monthly` DataFrame の1行＝ `field_id` × `year` × 1か月の平均値です。
この時点ではまだ「特徴量」ではなく、生の月別観測値です。

## 🔁 小さく検証する・変更に強い設計（軽い実践と説明）

いきなり全200筆を実行せず、段階的に確認しながら進めます。

```text
5筆
  ↓ 正しく動く？ → OK
20筆
  ↓ 値が妥当？ → OK
50筆
  ↓ 問題ない？ → OK
200筆（本番）
```

### なぜ小さく試すのか？

- **計算時間の短縮**だけでなく、**問題を早く発見するため**です
- GEE の `reduceRegions` は大量筆だと数分かかり、エラー時の再実行が重い
- 少数筆で「列名・値域・NaNの有無」を確認してからスケールアップ

### 変更に強い設計 = モジュール化

処理を以下の5モジュールに分けて考えると、後で修正・拡張が容易です：

```text
[1. 衛星データ取得]  S2/S1 コレクション、フィルタ、雲除去、軌道統一
[2. 前処理]         NDVI計算、dB確認、バンド選択
[3. 空間集計]       reduceRegions（月別平均）→ 月別DataFrame
[4. 特徴量生成]     波形特徴量14種の計算 → field_features.csv
[5. 判定・出力]     ルール/ML適用 → 判別マップ・優先リスト・GeoPackage
```

この分割なら、
- 対象地域を変える → 【1】【3】の範囲だけ変更
- 別作物（大豆・麦など）を対象にする → 【4】の特徴量定義だけ追加
- 判定手法をルールからMLに変える → 【5】だけ差し替え

### 技術 → 現場価値への翻訳

| 技術的な設計 | 自治体へのメリット（現場価値） |
|------------|--------------------------|
| 段階的な小規模検証 | 本番導入前に問題を発見し、失敗リスクを下げる |
| モジュール化（取得/集計/特徴量/判定/出力） | 別地域・別作物への展開がしやすい、追加開発コストを抑える |
| 共通関数化（物理ガード等） | 単位ミス・軌道混在・CRSズレを自動検知、属人化を防ぐ |

In [ ]:
# ── GEE: 月別平均画像を作り、reduceRegions で一筆ごとに空間集計 ──
def monthly_image(collection, band, year, mo):
    # 月初から「翌月初」まで。filterDate の終端は排他なので、これで its月の全日が入る。
    # （'-01' 〜 '-28' と書くと 28〜31日が毎月落ちる。S1 は12日回帰なので
    #   月によっては観測1回分がまるごと欠ける。）
    start = ee.Date.fromYMD(year, mo, 1)
    end   = start.advance(1, 'month')
    return (collection
            .filterDate(start, end)
            .select(band).mean().rename(f'{band}_{mo:02d}'))

BANDS = ['NDVI', 'VH_db', 'VV_db']

def reduce_fields(year):
    imgs = []
    for b in BANDS:
        for mo in months:
            imgs.append(monthly_image(s2_ndvi_all if b == 'NDVI' else s1_db_all, b, year, mo))
    img = ee.Image.cat(imgs).float()
    return img.reduceRegions(collection=fields, reducer=ee.Reducer.mean(),
                             scale=SCALE, crs=CRS, maxPixelsPerRegion=1e9)

SOURCE = 'GEE 実データ（Sentinel-2 / Sentinel-1）'
try:
    recs = []
    for y in years:
        for f in reduce_fields(y).getInfo()['features']:
            p = dict(f['properties'])
            p['year'] = y
            recs.append(p)
    monthly = pd.DataFrame(recs)
    band_cols = [f'{b}_{m:02d}' for b in BANDS for m in months]
    monthly = monthly[['field_id', 'year'] + band_cols]

    # 欠測の扱い: 「全24列そろっていること」を条件にすると、梅雨で1か月でも
    # 雲に覆われた年は圃場ごと消えてしまう（実測で 30行 → 8行 まで落ち、
    # 解析対象年 2024 が丸ごと消えた）。
    # 判別に直接使う 5月・8月だけを必須にし、波形特徴量は
    # 残った月から計算する（pandas の max/min は NaN を飛ばす）。
    KEY_COLS = ['NDVI_05', 'NDVI_08', 'VH_db_05', 'VH_db_08', 'VV_db_05', 'VV_db_08']
    nd_cols  = [f'NDVI_{m:02d}' for m in months]
    monthly = monthly.dropna(subset=KEY_COLS)                    # 5月・8月は必須
    monthly = monthly[monthly[nd_cols].notna().sum(axis=1) >= 4] # NDVI は8か月中4か月以上
    monthly = monthly.reset_index(drop=True)
    print(f'有効な筆×年: {len(monthly)} 行（5月・8月がそろっている行のみ）')
    assert len(monthly) > 0
except Exception as e:
    print(f'[フォールバック] GEE 抽出に失敗 → 教育用合成の月別データで続行: {e}')
    SOURCE = '教育用合成データ（フォールバック）'
    rng = np.random.default_rng(42)
    nf = 40
    fids = [f'field_{i:03d}' for i in range(nf)]
    types = rng.choice(['作付', '休耕', '森林'], size=nf, p=[0.5, 0.3, 0.2])
    # 月は months = [3, 4, 5, 6, 7, 8, 9, 10]。索引2が5月であることに注意。
    # 水稲の物理を素直に反映させる:
    #   5月 = 湛水 → 水面の鏡面反射で VH が急落（作付の決め手）
    #   7-8月 = 繁茂 → NDVI ピーク・VH は植生散乱で戻る
    # ここを崩すと、教材が教える疑似ラベル（vh_may ≦ -18 かつ ndvi_aug ≧ 0.70）に
    # 1件も当てはまらなくなるので、値を変えるときは必ず整合を確認すること。
    mbase = {
        #                 3月    4月    5月★   6月    7月    8月    9月   10月
        '作付': dict(nd=[0.25,  0.35,  0.30,  0.68,  0.78,  0.72,  0.55,  0.35],
                     vh=[-14.0, -15.0, -19.0, -15.0, -12.5, -12.0, -13.5, -14.5],
                     vv=[-10.0, -10.5, -14.0, -10.0, -7.0,  -6.5,  -7.5,  -8.5]),
        '休耕': dict(nd=[0.20, 0.21, 0.22, 0.23, 0.24, 0.23, 0.22, 0.20],
                     vh=[-13.0] * 8, vv=[-7.5] * 8),
        '森林': dict(nd=[0.78, 0.80, 0.81, 0.82, 0.83, 0.82, 0.80, 0.78],
                     vh=[-9.0] * 8, vv=[-5.5] * 8),
    }
    rows = []
    for i, (fid, t) in enumerate(zip(fids, types)):
        for y in years:
            row = {'field_id': fid, 'year': y}
            for j, mo in enumerate(months):
                row[f'NDVI_{mo:02d}'] = float(np.clip(rng.normal(mbase[t]['nd'][j], 0.03), 0, 1))
                row[f'VH_db_{mo:02d}'] = float(np.clip(rng.normal(mbase[t]['vh'][j], 0.6), -25, -5))
                row[f'VV_db_{mo:02d}'] = float(np.clip(rng.normal(mbase[t]['vv'][j], 0.6), -20, -3))
            rows.append(row)
    monthly = pd.DataFrame(rows)

print('データソース:', SOURCE)
print('形状（筆×年 × 月別バンド）:', monthly.shape)
print('圃場数:', monthly['field_id'].nunique(), '/ 年数:', monthly['year'].nunique())
monthly.head()


## 波形特徴量を計算する（Python 側）—「値」ではなく「波形の形」を見る

休耕地は「NDVI が低い」だけでなく、**「年間を通じて生育ピークが形成されない」** のが特徴です。
だから「単一時期の値」ではなく、**時系列波形から計算する特徴量**が判別に効きます。

- **Sentinel-2（NDVI）**: 年間最大値・最小値・振幅・5月/8月の値・**ピークの月**・**生育勾配**
- **Sentinel-1（VH）**: 5月/8月の値・**湛水期の急落（年間最小）**・季節振幅・**時間差分（8月−5月）**
- **VV/VH 比**: 5月と8月の比（構造（土壌・湛水）の情報）


In [ ]:
# ── Python: 月別データから「波形特徴量」を計算する ──
M  = months
nd = [f'NDVI_{m:02d}' for m in M]
vh = [f'VH_db_{m:02d}' for m in M]
vv = [f'VV_db_{m:02d}' for m in M]

m = monthly.copy()
for c in nd + vh + vv:
    m[c] = pd.to_numeric(m[c])

# ---- NDVI 波形 ----
m['ndvi_max']        = m[nd].max(axis=1)                  # 年間の生育ピーク
m['ndvi_min']        = m[nd].min(axis=1)
m['ndvi_amplitude']  = m['ndvi_max'] - m['ndvi_min']      # 生育の振幅（作付で大）
m['ndvi_may']        = m['NDVI_05']
m['ndvi_aug']        = m['NDVI_08']
m['ndvi_max_month']  = m[nd].idxmax(axis=1).str.extract(r'(\d+)$').astype(int)  # ピークの月
# 5月からピーク月までの「1か月あたりの平均生育速度」。
# ピークが5月以前（休耕地は3〜4月に来ることがある）は「5月以降に伸びていない」
# ことを意味するので 0 とする。
# ※ 分母を1にクリップしてはいけない。ピークが4月なら (4-5)=-1 → 1 にされ、
#   「1か月で急落した」という物理的に誤った勾配になってしまう。
_months_to_peak      = (m['ndvi_max_month'] - 5)
m['growth_slope']    = np.where(_months_to_peak >= 1,
                                (m['ndvi_max'] - m['ndvi_may']) / _months_to_peak.where(_months_to_peak >= 1, 1),
                                0.0)

# ---- VH 波形（SAR） ----
m['vh_may']          = m['VH_db_05']
m['vh_aug']          = m['VH_db_08']
m['vh_min']          = m[vh].min(axis=1)                  # 湛水期の急落（作付で低い）
m['vh_amplitude']    = m[vh].max(axis=1) - m[vh].min(axis=1)   # 季節振幅
m['vh_temporal_diff']= m['vh_aug'] - m['vh_may']           # 5月→8月の変化（作付で正に大きい）

# ---- VV/VH 比（dB では「差」を取る） ----
# dB は対数なので、線形の比 VV/VH は dB では「引き算」になる:
#     10*log10(VV_linear / VH_linear) = VV_db - VH_db
# dB 同士を割り算しても物理的な意味はない（VH が 0 に近いと発散し、
# 両方が負値なので符号も直感と逆になる）。単位は dB。
m['vv_vh_may']       = m['VV_db_05'] - m['VH_db_05']
m['vv_vh_aug']       = m['VV_db_08'] - m['VH_db_08']

FEATURES = ['ndvi_max', 'ndvi_min', 'ndvi_amplitude', 'ndvi_may', 'ndvi_aug',
            'ndvi_max_month', 'growth_slope', 'vh_may', 'vh_aug', 'vh_min',
            'vh_amplitude', 'vh_temporal_diff', 'vv_vh_may', 'vv_vh_aug']
field_features = m[['field_id', 'year'] + FEATURES].copy()
field_features.head(10)


## ✅ Check 2

この行は最終的に何になりますか？

> **1つの圃場の特徴量ベクトル（1筆＝1行）**

`field_features` DataFrame の1行＝ `field_id` × `year` の組み合わせで、
14種類の波形特徴量（`ndvi_max`, `ndvi_amplitude`, `vh_min` など）が並んでいます。
この表が W8 の判別・精度評価の入力になります。

## ✅ Check 3

自治体職員はこの CSV をどう使うでしょうか？

> **要確認圃場の抽出、現場巡回の優先順位付け、作付状況の把握**

- `ndvi_amplitude` が小さい圃場 → 休耕の疑い → 現地確認リストに入れる
- `vh_min` が湛水期に急落していない圃場 → 作付でない可能性 → 優先確認
- 確信度スコア（W6で学んだ `predict_proba` の考え方）を後から付与して、
  「グレーゾーン（要確認）」だけを職員に渡せば、巡回作業を大幅に削減できる

## 🎯 技術を現場価値に翻訳する（ミニワーク）

以下の3項目について、**非専門家（自治体職員）に伝わる言葉**で
メリットを1〜2文ずつ書いてみてください。

```python
# ここに記入してください（W7 Exercise Log の③として提出）

translation = {
    "大量処理（GEEで多数圃場を一括処理）":
        "これまで職員が1筆ずつ確認していた作業を、広域にまとめて処理し、"
        "確認が必要な圃場だけを絞り込めます。",
    "変更に強い設計（モジュール化）":
        "対象地域や作物が変わっても、モジュール単位で差し替えられるため、"
        "新規導入時の開発工数を大幅に削減できます。",
    "小規模検証・再実行可能な設計":
        "少数筆で動作確認してから全体展開するため、本番環境での予期せぬ"
        "エラーや計算ミスを未然に防げます。"
}
```

> **ヒント:** 「技術的な機能」ではなく「職員の作業時間・コスト・業務負担・拡張性」
> に結びつける言葉を選びましょう。これは合宿の自治体向け提案スライドで再利用します。

## 「1筆×1年=1行」の解析データセットを完成させる

これが W7 の成果物です。**1行 = 1筆（圃場）× 1年** の表に、判別に使える特徴量が並んでいます。
次の W8 ではこの表に判別ルール／機械学習を適用し、目視検証セットで精度を測ります。


In [ ]:
# ── 解析データセットの保存（W8・合宿で読み込む） ──
print('サンプル数（筆×年）:', len(field_features), '/ 圃場数:', field_features['field_id'].nunique())
print('特徴量:', FEATURES)

field_features.to_csv('field_features.csv', index=False)
print('保存: field_features.csv')

# 圃場ジオメトリを 05（地図化）に渡すため GeoJSON 保存（小規模ならそのまま保存できる）
try:
    import json
    with open('fields.geojson', 'w') as f:
        json.dump(fields.getInfo(), f)
    print('保存: fields.geojson（筆ジオメトリ）')
except Exception as e:
    print('（fields.geojson の保存はスキップ）:', e)


## 途中結果の可視化 — 波形の「形」を見る

作付の筆は夏に**明確なピーク**を持つ、休耕の筆は**平坦**、という違いをプロットで確認します。
「値ではなく波形の形を見る」感覚をここで掴みます。


In [ ]:
# ── 代表的な筆の NDVI 波形をプロット ──
def plot_waveforms(fids, title='代表的な筆の NDVI 波形（年ごと）'):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for fid in fids:
        sub = monthly[monthly['field_id'] == fid]
        for _, r in sub.iterrows():
            ax.plot(M, [r[c] for c in nd], marker='o', label=f'{fid} ({int(r["year"])})')
    ax.set_xticks(M); ax.set_xticklabels([f'{m}月' for m in M])
    ax.set_xlabel('月'); ax.set_ylabel('NDVI'); ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

sample_fids = field_features['field_id'].unique()[:4]
plot_waveforms(sample_fids)

print('''考察のヒント:
- 作付（水田）: 5月の湛水直後から上昇し、8月にピーク → 振幅が大きい
- 休耕: 通年平坦でピークなし（ndvi_amplitude が小さい）
- 森林: NDVI が常に高い（振幅は小さいが値は大きい）
''')


---

## まとめ（W8 への接続）

- **Zonal Statistics（`reduceRegions`）** で「画像」を「**1筆×1年=1行の表**」に変換しました
- **波形特徴量**（振幅・ピークの月・生育勾配・VHの季節振幅・VV/VH比）を作りました
  - キーワードは「**値ではなく波形の形**」「休耕地は生育ピークが形成されない」
- 保存物: `field_features.csv`（特徴量表）／`fields.geojson`（筆ジオメトリ）

> **問い: この `field_features.csv` は本当に信頼できるのか？**

**次の W8:** この表に判別ルールまたは機械学習を適用し、
「疑似ラベル」と「目視検証セット」の違いを踏まえて **F1 / Precision / Recall / IoU** で本当の精度を測ります。
**次の W8:** 閾値感度分析で「なぜこの閾値か」を根拠づけます。
**次の W8:** 判別結果を地図（緑=作付 / 赤=休耕 / 灰=要確認）と「要確認優先リスト」にして自治体に説明します。

### W7 の提出物（ミニマム3点・提出先：チームの Drive または camp-* ブランチ・任意）

1. **`field_features.csv`** を作成する
2. **波形特徴量の確認メモ（3点）** を書く
   - 実際に使った特徴量を3つ選び、「何を表すか」「作付/休耕とどう関係するか」「実データでどう見えたか」
3. **技術→現場価値の翻訳（3行）** を書く
   - 上記ミニワークの3項目（大量処理・モジュール化・小規模検証）の翻訳文

※ `validation_labels.csv`・精度評価・閾値感度分析・自治体向けスライドは W8 で扱います。